![banner](../../../docs/figs/brand_identity/banner.png)

# Week 7: Document Parsing with LLMs

## Lesson overview

Use a language model to process PDF documents: prepare searchable content, summarize it, answer questions, and organize information from multiple documents. The inputs are course PDFs; the outputs are text answers and a summary table.

**By the end, you can:** follow the document-processing steps; check whether answers are supported by the source text; and inspect a table combined from multiple files.

**Before you begin:** You need configured DeepSeek API credentials, an internet connection, and the course PDF files. Model answers may be wrong, so check the source text. Call cells send document content to an external service.

<p align="center"><img src="../../../docs/figs/energy_plan_flow.svg" alt="Technical PDFs become searchable passages; a researcher retrieves evidence, an LLM drafts consistent fields, and the notes form a comparison table" width="100%"><br><em>Figure 1. The Starter Kit workflow connects PDF preparation, retrieval, structured evidence notes, and a comparison table.</em></p>

**Use the figure as a checklist:** Name the input and output of each stage before running its cells. A retrieved passage is evidence for a draft extraction, while the original PDF remains the source to check.


## Goal and complete document workflow

This notebook demonstrates a literature-review support workflow: turn several PDF energy plans into comparable fields and a table. The process is **LLM setup → PDF extraction and chunking → retrieval → information extraction → table construction → post-processing and comparison**.

The source PDFs are in the repository Starter Kit data folder. Live LLM calls require the configured DeepSeek credential and internet. Generated text and CSV files are written to `output/` relative to the notebook working directory; review them before using them as evidence because extraction can omit or misstate source details.


## Why this example matters

Technical PDF collections can hide comparable facts in different sections and wording. This tutorial uses retrieval and structured extraction to create draft notes that a researcher can trace back to the original documents.

**Inputs and requirements:** Repository PDFs, document-processing packages, and configured DeepSeek access for live calls. A passage is a short source-text unit; retrieval selects passages relevant to a question.

**Route through the lesson:** Prepare PDF text → build retrieval → ask questions → extract the same fields per document → compare table rows against source PDFs.

**As you work:** Before each model or plot cell, predict one result. After it runs, describe one observation, identify its source, and name one limitation.


In [1]:
from ccai9012 import llm_utils
from ccai9012.paths import STARTER_KITS_DIR

/opt/anaconda3/envs/ccai9012_check/lib/python3.12/site-packages/torch/amp/autocast_mode.py:270: UserWarning: User provided device_type of 'cuda', but CUDA is not available. Disabling
  warnings.warn(


In [ ]:
from pathlib import Path
from ccai9012.paths import ensure_output_dir, WEEKLY_SCRIPTS_DIR
OUTPUT_DIR = ensure_output_dir(Path.cwd())
print(f"Intermediate summaries and CSV files will be saved to {OUTPUT_DIR}")


In [2]:
import pandas as pd
import os

## Initialize and test the LLM

Test the model connection first. Later cells send document content to the model service; make sure you are allowed to process these files.

In [3]:
# Initialize LLM
api_key = llm_utils.get_deepseek_api_key()
llm = llm_utils.initialize_llm()

Enter your DEEPSEEK_API_KEY:  ········


In [4]:
# Test connection
test = ["Is 9.9 or 9.11 bigger?"]
llm_utils.ask_llm(test)


📌 Prompt:
['Is 9.9 or 9.11 bigger?']

Let's compare the two numbers:  

**9.9** means \( 9 + \frac{9}{10} = 9.90 \)  
**9.11** means \( 9 + \frac{11}{100} = 9.11 \)  

Now compare digit by digit:  
- Both have 9 before the decimal.  
- Tenths digit: 9.9 has **9** in the tenths place, 9.11 has **1** in the tenths place.  

Since \( 9 > 1 \), we don't need to check further.  

\[
9.90 > 9.11
\]

**Answer:** 9.9 is bigger than 9.11.



## Sparse and embedding the document

After building the retriever, check the number of input files and the text chunks. The retriever finds relevant passages when you ask a question.

**Pause and predict:** If a relevant claim falls outside the retrieved chunks, can the LLM reliably extract it? After running, inspect a retrieved passage and identify its source document before reading the model summary.


In [9]:
retriever = llm_utils.build_pdf_retriever(
    pdf_path=str(STARTER_KITS_DIR / "2_llm_structure_output" / "pdf_extraction" / "data" / "1526439.pdf"),
    embedding_model_name="BAAI/bge-base-en-v1.5"
)

## Summarize the document

Ask the model to summarize the document. Spot-check important sentences in the source to make sure the summary has not missed conditions or added unsupported details.

In [10]:
summary = llm_utils.run_qa_chain(
    query="Please summarize the location, main objectives, actions, and stakeholders described in this energy plan document.",
    retriever=retriever,
    llm=llm,
    return_sources=True,
    save_path=str(OUTPUT_DIR / "summary.txt")
)


--- Final Answer ---
Based on the provided context, here is a summary of the location, main objectives, actions, and stakeholders described in the Aniak Energy Action Plan 2019.

**Location:**
The project is focused on the community of **Aniak, Alaska**. The plan specifically targets seven tribal buildings owned by the Aniak Traditional Council.

**Main Objectives:**
The primary goal of the project is to create an **Energy Action Plan** for tribal buildings with the aim of making them:
*   Safer
*   More comfortable
*   More energy-efficient
The plan is designed to serve as a foundational document to guide future building retrofit projects.

**Key Actions (Project Steps):**
The project followed a structured procedure:
1.  **Project Initiation:** Meeting with the Tribe to discuss goals and select buildings.
2.  **Data Collection:** Gathering baseline energy, occupancy, and maintenance data for each building.
3.  **Monitoring Plan:** Creating a plan to track building improvements over t

## Ask specific question

Ask a question that can be checked directly in the document, then see whether the answer cites or summarizes relevant content.

In [45]:
answer = llm_utils.run_qa_chain(
    query="Give detailed description of the responsibility of Cold Climate Housing Research Center (CCHRC)",
    retriever=retriever,
    llm=llm,
    return_sources=True,
    save_path=str(OUTPUT_DIR / "question.txt")
)


--- Final Answer ---
The **Cold Climate Housing Research Center (CCHRC)** plays a key role in supporting energy efficiency and renewable energy projects in cold climates, particularly in communities like Aniak, Alaska. Based on the provided context, here are the detailed responsibilities of CCHRC in the **Aniak Energy Action Plan 2019**:

### **Primary Responsibilities:**
1. **Project Coordination & Planning**  
   - Collaborates with **Energy Audits of Alaska** to develop an **Energy Action Plan** for the Aniak Tribe.  
   - Helps the Tribe meet grant requirements (e.g., writing quarterly progress reports, final reports, and outreach materials).  

2. **Energy Audits & Assessments**  
   - Works with **Energy Audits of Alaska** to conduct **on-site energy assessments** of buildings in Aniak.  
   - Collects **baseline data** (energy use, building conditions, occupant comfort) before audits.  

3. **Data Management & Reporting**  
   - Stores **baseline data** on CCHRC’s server and ma

## Extract information from multiple documents for comparison

Extract the same fields from multiple files. Check the fields and contents for each result before combining them.

In [11]:
from langchain.prompts import PromptTemplate

structured_prompt = PromptTemplate.from_template(
"""
Given the following document text, extract key information and output a markdown table with columns:

| Location | Main Objectives | Key Actions | Stakeholders | Timeline |
|----------|-----------------|-------------|--------------|----------|

Use exact information from the text; if any info is missing, write "N/A".

Context:
{context}

Question:
{question}
"""
)

In [27]:
folder_path = STARTER_KITS_DIR / "2_llm_structure_output" / "pdf_extraction" / "data"
save_csv_path = str(OUTPUT_DIR / "multiple_comparison.csv")
query_text = "Extract key information from this document."
results = []

for fname in sorted(os.listdir(folder_path)):
    if not fname.lower().endswith(".pdf"):
        continue
    pdf_path = os.path.join(folder_path, fname)
    print(f"Processing {pdf_path} ...")
    retriever = llm_utils.build_pdf_retriever(pdf_path)
    extracted_text = llm_utils.run_qa_chain(
        query=query_text, retriever=retriever, llm=llm,
        prompt_template=structured_prompt, return_sources=False,
    )
    results.append({"pdf_path": fname, "extracted_text": extracted_text})

df = pd.DataFrame(results)
df.to_csv(save_csv_path, index=False, encoding="utf-8-sig")
print(f"Saved results to {save_csv_path}")

Processing data/1526439.pdf ...

--- Final Answer ---
Based on the provided text, here is the extracted key information in the requested markdown table format.

| Location | Main Objectives | Key Actions | Stakeholders | Timeline |
|----------|-----------------|-------------|--------------|----------|
| Aniak | Improve building condition; Decrease energy use; Determine if future retrofit projects are successful. | Check fuel and electrical use monthly; Survey buildings for layout, insulation, condition, and safety; Collect and analyze baseline data; Maintain a maintenance notebook; Monitor data related to seasonal energy use and building operation. | Contractors; Traditional Council; Project staff; Tribal finance director; Tribal building maintenance manager; Occupants; Cold Climate Housing Research Center (CCHRC) | Monthly (energy bill analysis); Period just prior to the energy audit (for baseline data); Mid-2019 (time of publication) |
Processing data/1526994.pdf ...

--- Final Answe

## Combine the single-document results

After combining the results, check that each row still represents one file and look for missing or duplicate entries.

**Read the table:** Pick one row and trace a field back through the model answer and retrieved passage to the source PDF. Note any empty or uncertain field instead of filling it from memory.


In [23]:
merged_df_list = []
df = pd.read_csv(OUTPUT_DIR / "multiple_comparison.csv")

for i, md_text in enumerate(df["extracted_text"]):
    try:
        parsed = llm_utils.parse_markdown_table(md_text)
        parsed["source_doc"] = df.loc[i, "pdf_path"] if "pdf_path" in df.columns else f"doc_{i+1}"
        merged_df_list.append(parsed)
    except Exception as e:
        print(f"Failed to parse doc {i+1}: {e}")

final_df = pd.concat(merged_df_list, ignore_index=True)
final_df.to_csv(OUTPUT_DIR / "merged_energy_policy.csv", index=False)
final_df

In [25]:
final_df

,Location,Main Objectives,Key Actions,Stakeholders,Timeline,source_doc
0,Aniak,Improve building condition; Decrease energy us...,Check fuel and electrical use monthly; Survey ...,Contractors; Traditional Council; Project staf...,Monthly (energy bill analysis); Period just pr...,1526439.pdf
1,Kwigillingok,Improve building condition; Decrease energy us...,Survey buildings; Analyze energy bills; Mainta...,I.R.A. Council; Project staff; Tribal finance ...,N/A,1526994.pdf
2,Atmautluak,Create a data monitoring plan for each buildin...,1. Meet to discuss project goals. ; 2. Collect...,"Tribe, CCHRC (Cold Climate Housing Research Ce...",May 2018,1527001.pdf
3,N/A,- Improve building condition; - Decrease energ...,- Survey buildings; - Analyze energy bills; - ...,- I.R.A. Council; - Contractors; - Service pro...,N/A,1527003.pdf


## Is LLM able to further analysis the combined summary?

Treat the combined result as a draft to verify; a model summary is not an analysis finding.

In [27]:
summary_prompt = PromptTemplate.from_template(
    """
    Below are extracted policy tables from multiple documents.
    Compare common energy policy goals, distinctive actions, stakeholders,
    timelines, and differences. Keep the answer concise and structured.

    Tables:
    {context}
    """
)

from langchain.chains import LLMChain

all_tables = "\n\n".join(df["extracted_text"].dropna().tolist())
llm_chain = LLMChain(prompt=summary_prompt, llm=llm)
summary = llm_chain.invoke({"context": all_tables})
print(summary["text"])
with open(OUTPUT_DIR / "energy_policy_summary.txt", "w", encoding="utf-8") as f:
    f.write(summary["text"])

Based on the provided policy tables for the Alaskan communities of Aniak, Kwigillingok, and Atmautluak, here is a comparative summary.

### Common Energy Policy Goals
The communities share nearly identical primary objectives, focusing on practical, immediate improvements to building efficiency and energy management.
*   **Improve Building Condition:** All plans prioritize assessing and upgrading the physical state of buildings, including insulation, siding, and safety.
*   **Decrease Energy Use:** A universal goal is to reduce consumption of fuel and electricity to lower costs and increase sustainability.
*   **Data-Driven Planning:** Each project emphasizes the critical role of collecting and analyzing baseline data (e.g., from energy bills and building surveys) to inform future retrofit projects and measure success.

### Regions with the Most Comprehensive Plans
While all plans are focused, Atmautluak's project outline is the most structured and comprehensive.
*   **Atmautluak** has 

## What to take away

Retrieval helps a model find relevant parts of long documents, but answers still need to be checked against the source. **Try next:** Ask another question or compare documents. **Limit:** Incorrect chunks or missing passages can affect answers.